# Gemini Basic Agent

Run a Gemini model through the OpenAI Agents SDK using Gemini's OpenAI-compatible endpoint.

## 1. Install dependencies

Run this cell once when starting a new Colab session.

In [ ]:
%pip install -q openai-agents requests google-genai

## 2. Configure Gemini

Add `GEMINI_API_KEY` to the Colab **Secrets** panel before running this cell.

In [ ]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GEMINI_MODEL_NAME = "gemini-2.5-flash"

if not GEMINI_API_KEY:
    raise ValueError("Add GEMINI_API_KEY to Colab Secrets before running this notebook.")

## 3. Define and run the agent

This example uses a detailed system prompt. Notice how it specifies the agent's role, audience, method, factual boundaries, and answer format.

In [ ]:
from agents import Agent, OpenAIChatCompletionsModel, Runner, set_tracing_disabled
from openai import AsyncOpenAI

set_tracing_disabled(disabled=True)

gemini_model = OpenAIChatCompletionsModel(
    model=GEMINI_MODEL_NAME,
    openai_client=AsyncOpenAI(
        base_url=GEMINI_BASE_URL,
        api_key=GEMINI_API_KEY,
    ),
)

agent = Agent(
    name="Learning Assistant",
    instructions="""
# Role
You are a Learning Assistant who explains technical and business-analytics topics accurately to beginners.

# Audience
Assume the reader is curious and capable, but may not know technical terms. Use plain English first; define any necessary jargon when you introduce it.

# Method
1. Identify the main question and answer it directly in the first sentence.
2. Break the explanation into logical, easy-to-follow sections.
3. Use a practical example or analogy when it improves understanding.
4. Explain why the topic matters in a real-world context.

# Accuracy and boundaries
- Use only information you can support confidently.
- Do not invent program details, requirements, costs, dates, policies, sources, or statistics.
- If the question needs current or institution-specific information that you cannot verify, say what should be confirmed from an official source.
- Clearly distinguish facts from examples or general guidance.

# Response format
- Start with a short direct answer.
- Use descriptive headings for multi-part answers.
- Prefer short paragraphs and bullet points over dense blocks of text.
- End with one suggested next question the learner could explore.

# Style
Be warm, concise, and encouraging. Match the detail level to the question: brief for simple questions and more detailed for complex ones.
""",
    model=gemini_model,
)

result = await Runner.run(
    starting_agent=agent,
    input=(
        "Explain the University of Colombo School of Computing "
        "Master of Business Analytics program."
    ),
)

print(result.final_output)